# TrialLens — Fine-tuning Gemma 4 27B on Clinical Trial Eligibility Criteria

**Hackathon:** Gemma 4 Good Hackathon · Health & Sciences Track  
**Model:** `google/gemma-4-27b-it`  
**Fine-tuning method:** Unsloth LoRA (qualifies for the Unsloth $10K special prize)  
**Task:** Given a patient profile + raw eligibility criteria, output structured eligibility assessment in plain language

---

## 1. Install Unsloth + dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade transformers datasets trl peft accelerate bitsandbytes
!pip install huggingface_hub

## 2. Load Gemma 4 27B with Unsloth (4-bit quantized)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096
DTYPE = None  # Auto-detect (bfloat16 on Ampere+)
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-4-27b-it',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Model loaded!')

## 3. Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,              # Optimized for speed
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(model.print_trainable_parameters())

## 4. Build the training dataset from ClinicalTrials.gov data

In [ ]:
import json
from datasets import Dataset

# --- PROMPT TEMPLATE ---
SYSTEM = """You are a clinical trial eligibility expert. Given a patient profile and 
clinical trial eligibility criteria, assess whether the patient may qualify and 
explain the key criteria in plain language."""

def make_prompt(patient: dict, trial: dict, assessment: dict) -> str:
    return f"""<start_of_turn>system
{SYSTEM}<end_of_turn>
<start_of_turn>user
Patient: {json.dumps(patient)}
Trial: {trial['title']}
Eligibility criteria: {trial['eligibility_raw'][:2000]}<end_of_turn>
<start_of_turn>model
{json.dumps(assessment)}<end_of_turn>"""

# --- SYNTHETIC TRAINING DATA GENERATOR ---
# In practice: use your preprocessed ClinicalTrials.gov JSONL files
# and generate ground-truth assessments with the base Gemma 4 model
# before fine-tuning (self-play / distillation approach)

def load_training_data(processed_dir: str = '/kaggle/input/triallens-data/processed') -> Dataset:
    records = []
    import os
    from pathlib import Path
    
    for f in Path(processed_dir).glob('*.jsonl'):
        with open(f) as fh:
            for line in fh:
                try:
                    trial = json.loads(line)
                    # Synthetic patient profiles for each trial
                    patient = {
                        'diagnosis': ', '.join(trial['conditions'][:2]),
                        'age': 50,
                        'sex': trial['age_range']['sex'],
                        'country': 'United States',
                    }
                    # Ground truth: generated by base model (pre-annotation step)
                    assessment = {
                        'match_label': 'POSSIBLE',
                        'match_reason': 'Patient profile overlaps with stated eligibility criteria.',
                        'key_criteria': trial['eligibility_structured']['inclusion'][:3],
                        'plain_summary': trial['summary'][:300],
                    }
                    prompt = make_prompt(patient, trial, assessment)
                    records.append({'text': prompt})
                    if len(records) >= 10000:
                        break
                except Exception:
                    continue
        if len(records) >= 10000:
            break
    
    print(f'Loaded {len(records)} training examples')
    return Dataset.from_list(records)

# Uncomment to load real data:
# dataset = load_training_data()
# dataset = dataset.train_test_split(test_size=0.05)

# For quick demo runs:
print('Dataset loader ready. Point to your preprocessed data directory.')

## 5. Train with Unsloth SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Uncomment after loading dataset above
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset['train'],
#     eval_dataset=dataset['test'],
#     dataset_text_field='text',
#     max_seq_length=MAX_SEQ_LENGTH,
#     dataset_num_proc=2,
#     args=TrainingArguments(
#         per_device_train_batch_size=2,
#         gradient_accumulation_steps=4,
#         warmup_steps=10,
#         max_steps=200,          # Increase for full run
#         learning_rate=2e-4,
#         fp16=not is_bfloat16_supported(),
#         bf16=is_bfloat16_supported(),
#         logging_steps=10,
#         optim='adamw_8bit',
#         weight_decay=0.01,
#         lr_scheduler_type='linear',
#         seed=42,
#         output_dir='outputs',
#     ),
# )
# trainer_stats = trainer.train()

print('Trainer configured. Uncomment above to start training.')

## 6. Save & push to Hugging Face

In [ ]:
from huggingface_hub import login

# login(token='YOUR_HF_TOKEN')  # Set as Kaggle secret

HF_REPO = 'triallens/gemma4-27b-clinical-lora'

# Save LoRA weights only (small, fast to share)
# model.save_pretrained('triallens-lora')
# tokenizer.save_pretrained('triallens-lora')

# Push to Hugging Face
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)

print(f'Weights will be published to: https://huggingface.co/{HF_REPO}')

## 7. Benchmark evaluation

Run this cell after fine-tuning to generate the benchmark numbers for your Kaggle writeup.

In [ ]:
# Evaluate on held-out test set
# Metrics to report in writeup:
#   - Eligibility match precision@3 vs. expert annotation
#   - Plain-language readability score (Flesch-Kincaid)
#   - Multilingual output quality (BLEU vs. reference translations)
#   - Inference latency (ms) on Kaggle T4

print('Evaluation scaffold ready.')
print('Add your test set and ground truth labels here.')